In [154]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error

In [155]:
pd.set_option('display.max_columns', None)

In [156]:
import sys
import os

sys.path.append(os.path.abspath(".."))  
from Resources.properly_format_data import GetData

In [157]:
non_pa_list = ['PA', 'Barrels', 'LD', 'GB', 'FB', 'BABIP', 'wRC+', 'AVG+', 'BB%+', 'K%+', 'OBP+', 'SLG+', 'ISO+', 'BABIP+','LD+%', 'GB%+', 'FB%+', 'HR/FB%+', 'Pull%+', 'Cent%+', 'Oppo%+', 'Soft%+', 'Med%+', 'Hard%+']
pa_list = ['H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'HBP', 'SF', 'SH', 'SB', 'AB', 'IBB', 'SO']
labels = ['IDfg', 'Season', 'Name', 'Team', 'Age']
combined_list = pa_list + ['Barrels', 'LD', 'GB', 'FB']
plus_stats = ['wRC+', 'AVG+', 'BB%+', 'K%+', 'OBP+', 'SLG+', 'ISO+', 'BABIP+','LD+%', 'GB%+', 'FB%+', 'HR/FB%+', 'Pull%+', 'Cent%+', 'Oppo%+', 'Soft%+', 'Med%+', 'Hard%+']

In [158]:
data_set = GetData(2021, 2025, non_pa_list, pa_list, labels)

In [159]:
formatted_df = data_set.format_data_for_models(add_2026=True)

In [160]:
formatted_df = formatted_df.fillna(0)

In [161]:
save_for_league_averages = formatted_df.copy()

In [162]:
for stat in plus_stats:
    relative_rate = formatted_df[stat] - 100

    k = 100  
    weight = formatted_df['PA'] / (formatted_df['PA'] + k)

    formatted_df[stat] = weight * relative_rate

    for yr in [1, 2, 3]:

        relative_rate = formatted_df[f'{yr}Prev_{stat}'] - 100

        weight = (
            formatted_df[f'{yr}Prev_PA'] /
            (formatted_df[f'{yr}Prev_PA'] + k)
        )

        formatted_df[f'{yr}Prev_{stat}'] = weight * relative_rate

In [163]:
for stat in combined_list:
    league_average = np.sum(formatted_df[stat]) / np.sum(formatted_df['PA'])

    player_rate = formatted_df[stat] / formatted_df['PA']
    relative_rate = player_rate - league_average

    k = 200  # add tune per stat feature later
    weight = formatted_df['PA'] / (formatted_df['PA'] + k)

    formatted_df[stat] = weight * relative_rate

    for yr in [1, 2, 3]:

        player_rate = (
            formatted_df[f'{yr}Prev_{stat}'] /
            formatted_df[f'{yr}Prev_PA']
        )

        league_rate = (
            formatted_df[f'Prev_{yr}yr_League_Totals_{stat}'] /
            formatted_df[f'Prev_{yr}yr_League_Totals_PA']
        )

        relative_rate = player_rate - league_rate

        weight = (
            formatted_df[f'{yr}Prev_PA'] /
            (formatted_df[f'{yr}Prev_PA'] + k)
        )

        formatted_df[f'{yr}Prev_{stat}'] = weight * relative_rate


In [164]:
formatted_df = formatted_df.dropna(subset='1Prev_H')

In [165]:
formatted_df = formatted_df.fillna(0)

In [166]:
formatted_df_no_2025 = formatted_df[(formatted_df['Season'] != 2025) & (formatted_df['Season'] != 2026)]
features_2025_df = formatted_df[formatted_df['Season'] == 2025]
features_2026_df = formatted_df[formatted_df['Season'] == 2026]

In [167]:
past_rates = [
    col for col in formatted_df.columns
    if any(prefix in col for prefix in ['1Prev_', '2Prev_', '3Prev_'])
]


In [168]:
projection_df = formatted_df_no_2025[labels].copy()
projection_2025_df = features_2025_df[labels + ['PA']].copy()
projection_2026_df = features_2026_df[labels+['DC_pa']].copy()

X = formatted_df_no_2025[past_rates]
X_2025 = features_2025_df[past_rates]
X_2026 = features_2026_df[past_rates]

for stat in pa_list:

    y = formatted_df_no_2025[stat]

    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,     
        reg_alpha=0.0,       
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X, y)

    projection_df[f"{stat}_Proj"] = xgb_model.predict(X)
    projection_2025_df[stat] = xgb_model.predict(X_2025)
    projection_2026_df[stat] = xgb_model.predict(X_2026)


In [169]:
for stat in pa_list:
    league_average = np.sum(save_for_league_averages[stat])/np.sum(save_for_league_averages['PA'])
    projection_2025_df[stat] = projection_2025_df[stat] + league_average
    projection_2025_df[stat] = projection_2025_df[stat] * projection_2025_df['PA']
    projection_2026_df[stat] = projection_2026_df[stat] + league_average
    projection_2026_df[stat] = projection_2026_df[stat] * projection_2026_df['DC_pa']

In [170]:
for stat in pa_list:
    age_adj = np.where(
        projection_2025_df['Age'] > 29,
        1 / (1 + 0.003 * (projection_2025_df['Age'] - 29)),
        np.where(
            projection_2025_df['Age'] < 29,
            1 + 0.006 * (29 - projection_2025_df['Age']),
            1
        )
    )
    if stat ==  'SO' or stat == 'CS':
        age_adj = 1/age_adj

    projection_2025_df[stat] = projection_2025_df[stat]*age_adj

In [172]:
projection_2025_df['wOBA'] = ((.691*projection_2025_df['BB']) + (.722*projection_2025_df['HBP']) + (.882*projection_2025_df['1B']) + (1.252*projection_2025_df['2B']) + (1.584*projection_2025_df['3B']) + (2.037*projection_2025_df['HR']))/(projection_2025_df['AB'] + projection_2025_df['BB'] - projection_2025_df['IBB'] + projection_2025_df['SF'] + projection_2025_df['HBP'])

In [173]:
projection_2025_df.head()

,IDfg,Season,Name,Team,Age,PA,H,1B,2B,3B,HR,R,RBI,BB,HBP,SF,SH,SB,AB,IBB,SO,wOBA
44,10155,2025,Mike Trout,LAA,33.0,556.0,116.533219,69.863385,24.097100,2.782564,21.728654,69.214425,63.985698,58.499700,7.165112,3.137227,0.856790,18.016356,486.341171,2.753471,137.661029,0.336817
66,10200,2025,Tucker Barnhart,TEX,34.0,15.0,2.955612,2.021817,0.613141,0.060346,0.381324,1.673704,1.449405,1.340080,0.151965,0.103083,0.126057,0.213280,13.278815,0.047902,3.802465,0.300751
73,10231,2025,Jose Iglesias,SDP,35.0,343.0,86.007112,62.818963,15.292127,0.989245,5.832027,36.525781,31.765177,16.819184,5.558396,1.453737,1.008418,6.220976,318.160265,0.525790,59.477368,0.303498
82,10243,2025,Randal Grichuk,- - -,33.0,293.0,64.584840,40.393176,13.983474,0.873698,9.220870,34.010705,36.980615,20.910934,3.458808,2.059728,0.293379,2.144130,266.277151,1.047284,59.283557,0.309428
112,10324,2025,Marcell Ozuna,ATL,34.0,592.0,131.376848,80.098298,24.454960,0.884056,27.796164,70.818840,78.367861,51.715857,4.043237,3.159336,0.399284,3.845962,532.682285,1.969961,145.061567,0.335702


In [174]:
projection_2025_df.to_csv('../correct wOBA comparison/my_system.csv', index=False)